# 🩺 糖尿病患病風險評估 API 教學：實作 `/predict` 風險預測端點

> 對象：修習邏輯迴歸分類器 / 模型部署課程的學生
> 對應程式碼：`app.py` 第 44~85 行的 `predict_endpoint`
>
> 這份 Notebook 會帶你**一步步拆解**推論流程：從載入 `.joblib` 模型檔、StandardScaler 特徵標準化、呼叫 `predict_proba` 計算患病機率，到臨床風險分級與建議。

## 🎯 學習目標

- [ ] 理解推論階段必須使用與訓練階段**完全相同的 `StandardScaler`** 進行特徵縮放
- [ ] 掌握文字類別特徵（性別：'男生'/'女生'）轉為數值 (1/0) 的前處理方式
- [ ] 看懂 Pydantic 的 `PatientData` 欄位驗證範圍防呆機制（年齡 1~120、體重 20~250 等）
- [ ] 掌握呼叫 `model.predict_proba()` 提取患病機率（$P(Y=1|X)$）並進行三級風險切分
- [ ] 能用 `TestClient` 呼叫真實 API 端點並驗證合法與異常輸入

## 📌 背景：`/predict` 完整預測流程

```
病患臨床指標 [age, weight, blood_sugar, gender]
       │
       ▼
1. 性別轉換 (男生: 1, 女生: 0) ──> 轉為 2D 陣列 [[age, weight, sugar, gender_num]]
       │
       ▼
2. scaler.transform(...)      ──> 進行特徵標準化 (Z-Score)
       │
       ▼
3. model.predict_proba(...)   ──> 取患病機率 P (索引 1)
       │
       ▼
4. 臨床分級邏輯               ──> P >= 0.7: 極高風險 / P >= 0.4: 中度風險 / 低風險
       │
       ▼
回傳 PredictionResponse 結構化 JSON
```

In [ ]:
# ============================================
# 0. 載入套件與模型
# ============================================
import os, joblib
import numpy as np
from pprint import pprint

MODEL_PATH = "diabetes_model.joblib"
if not os.path.exists(MODEL_PATH):
    from train_save import train_and_save_model
    train_and_save_model(model_path=MODEL_PATH)

meta = joblib.load(MODEL_PATH)
model = meta["model"]
scaler = meta["scaler"]
print("✅ 成功載入模型與標準化器！")

---
## Part 1：手動測試單筆資料特徵預處理與推論

模擬一筆病患數據：45歲、體重75公斤、空腹血糖126 mg/dL、男生。

In [ ]:
age = 45.0
weight = 75.0
blood_sugar = 126.0
gender = "男生"

# 1. 特徵數值轉換
gender_num = 1 if gender == "男生" else 0
raw_features = np.array([[age, weight, blood_sugar, gender_num]])

# 2. 特徵標準化
scaled_features = scaler.transform(raw_features)

# 3. 計算機率
prob = float(model.predict_proba(scaled_features)[0, 1])
print(f"病患患病預估機率：{prob * 100:.2f}%")

---
## Part 2：定義 Pydantic 資料模型

使用 Pydantic 對病患數據實施嚴格的防呆機制。

In [ ]:
from pydantic import BaseModel, Field

class PatientData(BaseModel):
    age: float = Field(..., ge=1, le=120, description="病患年齡 (歲)", example=45)
    weight: float = Field(..., ge=20, le=250, description="病患體重 (公斤)", example=75.5)
    blood_sugar: float = Field(..., ge=40, le=400, description="空腹血糖 (mg/dL)", example=126.0)
    gender: str = Field(..., description="生理性別 ('男生' 或 '女生')", example="男生")

class PredictionResponse(BaseModel):
    risk_probability: float
    is_diabetic: int
    risk_level: str
    clinical_advice: str

print("✅ Pydantic 資料模型宣告完成！")

---
## Part 3：建立 FastAPI App 與 `/predict` 預測端點

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Diabetes Prediction API")

@app.post("/predict", response_model=PredictionResponse)
def predict_endpoint(patient: PatientData):
    gender_num = 1 if patient.gender == "男生" else 0
    raw_features = np.array([[patient.age, patient.weight, patient.blood_sugar, gender_num]])
    
    scaled_features = scaler.transform(raw_features)
    prob = float(model.predict_proba(scaled_features)[0, 1])
    pred = int(prob >= 0.5)
    
    if prob >= 0.7:
        risk_level = "🔴 極高風險 (High Risk)"
        advice = "空腹血糖與生化數據偏高，強烈建議盡速安排糖化血色素 (HbA1c) 檢測與新陳代謝科門診複診。"
    elif prob >= 0.4:
        risk_level = "🟡 中度風險 (Moderate Risk)"
        advice = "處於糖尿病前期或潛在風險群，建議注意飲食熱量控制並進行生活型態衛教。"
    else:
        risk_level = "🟢 低風險 (Low Risk)"
        advice = "生理指標目前落於良好範圍，請持續維持規律運動與健康飲食。"
        
    return PredictionResponse(
        risk_probability=round(prob, 4),
        is_diabetic=pred,
        risk_level=risk_level,
        clinical_advice=advice
    )

print("✅ /predict 端點建立完成！")

---
## Part 4：使用 `TestClient` 模擬呼叫 `/predict` 端點

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

test_payload = {
    "age": 55,
    "weight": 82.0,
    "blood_sugar": 145.0,
    "gender": "男生"
}

response = client.post("/predict", json=test_payload)
print("狀態碼：", response.status_code)
print("回應結果：")
pprint(response.json())